In [2]:
pip install optuna

  Using cached optuna-4.9.0-py3-none-any.whl.metadata (15 kB)
  Using cached alembic-1.18.5-py3-none-any.whl.metadata (7.2 kB)
  Using cached colorlog-6.10.1-py3-none-any.whl.metadata (11 kB)
  Using cached sqlalchemy-2.0.51-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (9.5 kB)
  Using cached tqdm-4.68.3-py3-none-any.whl.metadata (57 kB)
  Using cached mako-1.3.12-py3-none-any.whl.metadata (2.9 kB)
  Using cached greenlet-3.5.3-cp312-cp312-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata (3.8 kB)
Using cached optuna-4.9.0-py3-none-any.whl (425 kB)
Using cached alembic-1.18.5-py3-none-any.whl (264 kB)
Using cached sqlalchemy-2.0.51-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl (3.4 MB)
Using cached greenlet-3.5.3-cp312-cp312-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl (614 kB)
Using cached colorlog-6.10.1-py3-none-any.whl (11 kB)
Using cached mako-1.3.12-py3-none-any.whl (78 kB)
Using cached tqdm

In [ ]:
import json
import os
import signal
import subprocess
import time
from pathlib import Path
from typing import Dict, List, Optional

import optuna


# -------------------------------------------------
# Paths
# -------------------------------------------------
REPO_ROOT = Path("/workspace/FraudGT-main")
PYTHON = "/workspace/venvs/fraudgt/bin/python"
CFG_FILE = REPO_ROOT / "configs/BTC/BTC-full-GINE.yaml"
BASE_OUT_DIR = REPO_ROOT / "results_btc_optuna_GINE"

# Headless backend for notebook/container runs
BASE_ENV = os.environ.copy()
BASE_ENV["MPLBACKEND"] = "Agg"

# How often to check val/stats.json while training is running
POLL_SECONDS = 10

# If no validation file appears for too long, just keep training.
# Increase this if your first validation takes longer.
NO_STATS_WARNING_SECONDS = 300


# -------------------------------------------------
# Helpers
# -------------------------------------------------
def find_newest_val_stats_file(trial_out: Path) -> Optional[Path]:
    """
    Find the newest validation stats file under a trial directory.
    FraudGT usually writes something like:
        trial_x/.../val/stats.json
    """
    candidates = list(trial_out.rglob("val/stats.json"))
    if not candidates:
        return None
    return max(candidates, key=lambda p: p.stat().st_mtime)


def read_stats_json_lines(stats_file: Path) -> List[Dict]:
    """
    Read FraudGT stats.json as JSON lines.
    Each line is expected to be one JSON dictionary.
    """
    records = []

    if not stats_file.exists():
        return records

    with stats_file.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue

            try:
                record = json.loads(line)
            except json.JSONDecodeError:
                # File may be written while we are reading it.
                # Skip incomplete line and check again later.
                continue

            records.append(record)

    return records


def read_best_val_f1(trial_out: Path) -> float:
    """
    Find the best validation F1 under a trial directory.

    FraudGT writes stats.json as JSON lines, one dict per line.
    """
    stats_file = find_newest_val_stats_file(trial_out)
    if stats_file is None:
        raise FileNotFoundError(f"No val/stats.json found under {trial_out}")

    stats_list = read_stats_json_lines(stats_file)

    # Deduplicate by epoch
    deduped = []
    seen_epochs = set()

    for record in stats_list:
        epoch = record.get("epoch")

        if epoch is None:
            continue

        if epoch not in seen_epochs:
            deduped.append(record)
            seen_epochs.add(epoch)

    if not deduped:
        raise ValueError(f"No valid stats found in: {stats_file}")

    best = max(deduped, key=lambda x: x.get("f1", float("-inf")))

    if "f1" not in best:
        raise KeyError(f"'f1' not found in best stats entry: {best}")

    return float(best["f1"])


def stop_process_tree(process: subprocess.Popen) -> None:
    """
    Stop FraudGT process safely.

    start_new_session=True is used in Popen, so os.killpg can stop the
    child process group on Linux/RunPod.
    """
    if process.poll() is not None:
        return

    try:
        os.killpg(os.getpgid(process.pid), signal.SIGTERM)
        process.wait(timeout=30)
    except Exception:
        try:
            os.killpg(os.getpgid(process.pid), signal.SIGKILL)
        except Exception:
            process.kill()


def run_fraudgt_trial(
    trial: optuna.Trial,
    trial_out: Path,
    batch_size: int,
    lr: float,
    weight_decay: float,
    loss_weight: int,
) -> None:
    """
    Launch one FraudGT training run and monitor validation F1 for pruning.

    This works if FraudGT writes val/stats.json during training.
    If val/stats.json is written only at the end, pruning cannot happen early.
    """
    cmd = [
        PYTHON,
        "-m", "fraudGT.main",
        "--cfg", str(CFG_FILE),
        "--gpu", "0",
        "--repeat", "1",

        "out_dir", str(trial_out),
        "wandb.use", "False",

        "train.batch_size", str(batch_size),
        "optim.base_lr", str(lr),
        "optim.weight_decay", str(weight_decay),
        "model.loss_fun_weight", f"[1, {loss_weight}]",

        "optim.max_epoch", "100",

        # Your current test settings
        "train.iter_per_epoch", "64",
        "val.iter_per_epoch", "64",
    ]

    log_file = trial_out / "fraudgt_subprocess.log"
    trial_out.mkdir(parents=True, exist_ok=True)

    with log_file.open("w", encoding="utf-8") as log:
        process = subprocess.Popen(
            cmd,
            cwd=str(REPO_ROOT),
            stdout=log,
            stderr=subprocess.STDOUT,
            text=True,
            env=BASE_ENV,
            start_new_session=True,
        )

        reported_epochs = set()
        start_time = time.time()
        warned_no_stats = False

        try:
            while process.poll() is None:
                stats_file = find_newest_val_stats_file(trial_out)

                if stats_file is None:
                    elapsed = time.time() - start_time
                    if elapsed > NO_STATS_WARNING_SECONDS and not warned_no_stats:
                        print(
                            f"Trial {trial.number}: no val/stats.json yet after "
                            f"{NO_STATS_WARNING_SECONDS}s. Continuing..."
                        )
                        warned_no_stats = True

                    time.sleep(POLL_SECONDS)
                    continue

                records = read_stats_json_lines(stats_file)

                for record in records:
                    epoch = record.get("epoch")
                    f1 = record.get("f1")

                    if epoch is None or f1 is None:
                        continue

                    if epoch in reported_epochs:
                        continue

                    epoch = int(epoch)
                    f1 = float(f1)

                    reported_epochs.add(epoch)

                    # Report intermediate result to Optuna
                    trial.report(f1, step=epoch)

                    print(
                        f"Trial {trial.number} | "
                        f"epoch={epoch} | val_f1={f1:.6f}"
                    )

                    # Ask Optuna whether this trial should stop early
                    if trial.should_prune():
                        print(
                            f"Trial {trial.number} pruned at epoch {epoch} "
                            f"with val_f1={f1:.6f}"
                        )
                        stop_process_tree(process)
                        raise optuna.TrialPruned()

                time.sleep(POLL_SECONDS)

            returncode = process.wait()

            if returncode != 0:
                print(f"\nTrial {trial.number} failed.")
                print(f"See log file: {log_file}")
                raise RuntimeError("FraudGT run failed")

        except optuna.TrialPruned:
            raise

        except Exception:
            stop_process_tree(process)
            raise


# -------------------------------------------------
# Optuna objective
# -------------------------------------------------
def objective(trial: optuna.Trial) -> float:
    # Based on your previous result, batch_size=64 was good.
    # I include 64 here. Remove it only if you intentionally do not want it.
    batch_size = trial.suggest_categorical(
        "batch_size",
        [128, 256, 512, 1024, 2048]
    )

    lr = trial.suggest_float(
        "base_lr",
        1e-4,
        3e-3,
        log=True
    )

    weight_decay = trial.suggest_float(
        "weight_decay",
        1e-6,
        1e-4,
        log=True
    )

    loss_weight = trial.suggest_categorical(
        "loss_weight",
        [50, 75, 100, 105, 125]
    )

    trial_out = BASE_OUT_DIR / f"trial_{trial.number}"
    trial_out.mkdir(parents=True, exist_ok=True)

    run_fraudgt_trial(
        trial=trial,
        trial_out=trial_out,
        batch_size=batch_size,
        lr=lr,
        weight_decay=weight_decay,
        loss_weight=loss_weight,
    )

    val_f1 = read_best_val_f1(trial_out)

    print(
        f"Trial {trial.number} finished: "
        f"best_val_f1={val_f1:.6f} | "
        f"params={trial.params}"
    )

    return val_f1


# -------------------------------------------------
# Main
# -------------------------------------------------
if __name__ == "__main__":
    BASE_OUT_DIR.mkdir(parents=True, exist_ok=True)

    sampler = optuna.samplers.TPESampler(
        seed=42,
        n_startup_trials=8,
        multivariate=True,
    )

    # MedianPruner is simpler and safer for subprocess monitoring.
    # It prunes a trial if its intermediate score is worse than the median
    # of previous trials at the same epoch.
    pruner = optuna.pruners.MedianPruner(
        n_startup_trials=8,
        n_warmup_steps=10,
        interval_steps=5,
    )

    study = optuna.create_study(
        direction="maximize",
        sampler=sampler,
        pruner=pruner,
        study_name="btc_GINE_pruning",
    )

    study.optimize(
        objective,
        n_trials=30,
        gc_after_trial=True,
    )

    print("\nBest params:")
    print(study.best_params)

    print("\nBest value:")
    print(study.best_value)

    print("\nNumber of trials:")
    print(len(study.trials))

    print("\nTrial summary:")
    for t in study.trials:
        print(
            f"Trial {t.number} | "
            f"state={t.state} | "
            f"value={t.value} | "
            f"params={t.params}"
        )

/tmp/ipykernel_1003/2813589214.py:312: ExperimentalWarning: Argument ``multivariate`` is an experimental feature. The interface can change in the future.
  sampler = optuna.samplers.TPESampler(
[I 2026-06-29 17:41:24,211] A new study created in memory with name: btc_GINE_pruning


Trial 0 | epoch=0 | val_f1=0.011990
Trial 0 | epoch=3 | val_f1=0.049110
Trial 0 | epoch=7 | val_f1=0.047300
Trial 0 | epoch=11 | val_f1=0.079250
Trial 0 | epoch=15 | val_f1=0.103040
Trial 0 | epoch=19 | val_f1=0.117600
Trial 0 | epoch=23 | val_f1=0.145790
